# Agregace a spojování

Stáhneme si soubory, se kterými budeme pracovat.

Vytvoříme pandas DataFrame pro soubory s výsledky maturitní zkoušky.

In [28]:
import pandas

In [29]:
## Tak to pojďme vyčistit
u202 = pandas.read_csv("../../data/u202.csv").dropna()
u203 = pandas.read_csv("../../data/u203.csv").dropna()
u302 = pandas.read_csv("../../data/u302.csv").dropna()

## Spojení dat

Pojďme spojit naše tři tabulky dohromady.


- Tabulky spojíme pod sebe. 
- Budeme stále mít tři sloupce.
- Počet řádků bude odpovídat počtu řádků všech tří tabulek dohromady.
- _V SQL tomu odpovídá operace `UNION`._

Použijeme naše DataFrames a očistíme je o řádky s chybějícími hodnotami

**Funkce `concat`**


! Pozor: funkce rozbije index. 
<br>
<br>
Spojí totiž indexy tabulek za sebe např. 1 2 3 1 2 3 1 2 3. <br>
Pomůže nám parameter `ignore_index`

In [30]:
maturita = pandas.concat([u202, u203, u302])
maturita

,cisloStudenta,predmet,znamka,den
1,2,Dějepis,3.0,pá
2,3,Matematika,2.0,út
3,2,Společenské vědy,2.0,pá
4,4,Biologie,1.0,pá
5,5,Dějepis,1.0,po
6,6,Fyzika,2.0,čt
7,7,Dějepis,4.0,po
8,8,Matematika,2.0,po
10,10,Chemie,2.0,st
11,3,Chemie,5.0,út


In [31]:
maturita = pandas.concat([u202, u203, u302], ignore_index=True)
maturita

,cisloStudenta,predmet,znamka,den
0,2,Dějepis,3.0,pá
1,3,Matematika,2.0,út
2,2,Společenské vědy,2.0,pá
3,4,Biologie,1.0,pá
4,5,Dějepis,1.0,po
5,6,Fyzika,2.0,čt
6,7,Dějepis,4.0,po
7,8,Matematika,2.0,po
8,10,Chemie,2.0,st
9,3,Chemie,5.0,út


**Vytvoření nového sloupce**

Spojením tabulek jsme ztratili informace o čísle místnosti. 

Můžeme si ale číslo místnosti před spojením uložit do nového sloupečku.

In [32]:
u202["mistnost"] = "u202"
u203["mistnost"] = "u203"
u302["mistnost"] = "u302"

maturita = pandas.concat([u202, u203, u302], ignore_index=True)
maturita

,cisloStudenta,predmet,znamka,den,mistnost
0,2,Dějepis,3.0,pá,u202
1,3,Matematika,2.0,út,u202
2,2,Společenské vědy,2.0,pá,u202
3,4,Biologie,1.0,pá,u202
4,5,Dějepis,1.0,po,u202
5,6,Fyzika,2.0,čt,u202
6,7,Dějepis,4.0,po,u202
7,8,Matematika,2.0,po,u202
8,10,Chemie,2.0,st,u202
9,3,Chemie,5.0,út,u202


**Uložení tabulky do souboru**

In [33]:
# dataframe.to_csv('soubor.csv', index=False)

maturita.to_csv("maturita.csv", index=False)

Finální tabulku, kterou jsme právě vytvořili, si můžete pro kontrolu stáhnout zde: 

<https://kodim.cz/cms/assets/analyza-dat/python-data-1/python-pro-data-1/agregace-a-spojovani/pokrocile-upravy/maturita.csv>

---

## Propojení dat

- Tabulky můžeme spojit více způsoby. 
- _V SQL tomu odpovídá operace `JOIN`._
- Výsledná tabulka bude mít více sloupců.
- Počet řádků na typu propojení.


**Typy propojení**

![Druhy propojeni](../../img/type_of_join_operations.png)

**Přidáme další tabulku**

Naše výsledky byly anonymní. Pokud bychom ale chtěli vytisknout maturitní vysvědčení, potřebujeme k číslům studenta zjistit jejich jména. 

Jména najdeme v samostatné tabulce `studenti.csv`. Načtěme si jej jako `DataFrame`.

In [34]:
studenti = pandas.read_csv("../../data/studenti.csv")
studenti.head()

,cisloStudenta,jmeno
0,1,Jana Zbořilová
1,2,Lukáš Jurčík
2,3,Pavel Horák
3,4,Pavel Kysilka
4,5,Kateřina Novotná


U operace `JOIN` jsou důležité dvě věci:

- Podle jakého sloupce (nebo jakých sloupců) dvě různé tabulky propojujeme.
- Co udělat v případě, že pro nějaké řádky nemám ve druhé tabulce odpovídající hodnotu.

**Funkce `merge`**

Dokumentace: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.merge.html

In [35]:
propojeny_df = pandas.merge(u202, studenti)
propojeny_df.head()

,cisloStudenta,predmet,znamka,den,mistnost,jmeno
0,2,Dějepis,3.0,pá,u202,Lukáš Jurčík
1,2,Společenské vědy,2.0,pá,u202,Lukáš Jurčík
2,3,Matematika,2.0,út,u202,Pavel Horák
3,3,Chemie,5.0,út,u202,Pavel Horák
4,4,Biologie,1.0,pá,u202,Pavel Kysilka


Ve výchozím nastavení funkce merge() ponechá pouze řádky, které mají záznamy v obou tabulkách. 

V SQL bychom tuto operaci označili jako INNER JOIN.

In [36]:
"""
Pokud by například nějaký student nebyl uvedený v tabulce se studenty, jeho maturitní výsledek by zmizel. 
U nového DataFrame bychom tedy měli zkontrolovat, zda má propojeny_df stejný počet řádků jako u202.
"""

print(u202.shape)

print(propojeny_df.shape)

(13, 5)
(13, 6)


**Tabulka s předsedy maturitních komisí**

In [37]:
preds = pandas.read_csv("../../data/predsedajici.csv")
preds.head()

,datum,jmeno,den
0,20.5.2019,Marie Zuzaňáková,po
1,21.5.2019,Marie Zuzaňáková,út
2,22.5.2019,Petr Ortinský,st
3,23.5.2019,Petr Ortinský,čt
4,24.5.2019,Alena Pniáčková,pá


In [38]:
novy_propojeny_df = pandas.merge(propojeny_df, preds)
novy_propojeny_df

,cisloStudenta,predmet,znamka,den,mistnost,jmeno,datum


Tentokrát jsme příliš neuspěli, výsledný DataFrame je prázdný. 

Protože v obou DataFrame máme sloupec **jmeno**, v jednom případě však jde o jméno studenta a ve druhém o jméno předsedy komise. 

Musíme říct, že chceme data spojit pouze podle sloupce den.

In [39]:
novy_propojeny_df = pandas.merge(propojeny_df, preds, on=["den"])
novy_propojeny_df

,cisloStudenta,predmet,znamka,den,mistnost,jmeno_x,datum,jmeno_y
0,2,Dějepis,3.0,pá,u202,Lukáš Jurčík,24.5.2019,Alena Pniáčková
1,2,Společenské vědy,2.0,pá,u202,Lukáš Jurčík,24.5.2019,Alena Pniáčková
2,4,Biologie,1.0,pá,u202,Pavel Kysilka,24.5.2019,Alena Pniáčková
3,3,Matematika,2.0,út,u202,Pavel Horák,21.5.2019,Marie Zuzaňáková
4,3,Chemie,5.0,út,u202,Pavel Horák,21.5.2019,Marie Zuzaňáková
5,6,Fyzika,2.0,čt,u202,Marie Krejcárková,23.5.2019,Petr Ortinský
6,10,Chemie,2.0,st,u202,Miroslav Bednář,22.5.2019,Petr Ortinský
7,10,Dějepis,5.0,st,u202,Miroslav Bednář,22.5.2019,Petr Ortinský
8,11,Matematika,1.0,st,u202,Ivana Dvořáková,22.5.2019,Petr Ortinský
9,12,Biologie,4.0,st,u202,Lenka Jarošová,22.5.2019,Petr Ortinský


**Zmizely nám řádky!**

To znamená, že funkce `merge()` **nenašla pro všechna zkoušení odpovídajícího předsedu**. 

<br>

Zkusme nyní říct funkci `merge()`, aby nám zachovala v prvním DataFrame ty řádky, pro které nenajde odpovídající záznam. 

Této operaci se v jazyce SQL říká **LEFT OUTER JOIN**. 

My ho provede tak, že funkci `merge()` jako parametr `how` zadáme hodnotu `left`.

In [40]:
novy_propojeny_df = pandas.merge(
    propojeny_df, preds, on=["den"], how="left"
)  # how = "outer"
novy_propojeny_df

,cisloStudenta,predmet,znamka,den,mistnost,jmeno_x,datum,jmeno_y
0,2,Dějepis,3.0,pá,u202,Lukáš Jurčík,24.5.2019,Alena Pniáčková
1,2,Společenské vědy,2.0,pá,u202,Lukáš Jurčík,24.5.2019,Alena Pniáčková
2,3,Matematika,2.0,út,u202,Pavel Horák,21.5.2019,Marie Zuzaňáková
3,3,Chemie,5.0,út,u202,Pavel Horák,21.5.2019,Marie Zuzaňáková
4,4,Biologie,1.0,pá,u202,Pavel Kysilka,24.5.2019,Alena Pniáčková
5,5,Dějepis,1.0,po,u202,Kateřina Novotná,NaN,NaN
6,6,Fyzika,2.0,čt,u202,Marie Krejcárková,23.5.2019,Petr Ortinský
7,7,Dějepis,4.0,po,u202,Vasil Lácha,NaN,NaN
8,8,Matematika,2.0,po,u202,Alexey Opatrný,NaN,NaN
9,10,Chemie,2.0,st,u202,Miroslav Bednář,22.5.2019,Petr Ortinský


Zkusme si zobrazit ty řádky, které se nepodařilo propojit. 

Poznáme je tak, že mají prázdný sloupec datum.

Z nějakého důvodu nám nefunguje propojení v případě, že ve sloupci den je hodnota po. 

In [41]:
novy_propojeny_df[novy_propojeny_df["datum"].isnull()]

,cisloStudenta,predmet,znamka,den,mistnost,jmeno_x,datum,jmeno_y
5,5,Dějepis,1.0,po,u202,Kateřina Novotná,NaN,NaN
7,7,Dějepis,4.0,po,u202,Vasil Lácha,NaN,NaN
8,8,Matematika,2.0,po,u202,Alexey Opatrný,NaN,NaN


In [42]:
# pondeli v tabulce preds obsahuje mezeru navic
print(propojeny_df["den"].unique())
print(preds["den"].unique())

['pá' 'út' 'po' 'čt' 'st']
['po ' 'út' 'st' 'čt' 'pá']


**funkce `strip()`**

z řetězce odstraní mezery (a další bílé znaky) na začátku a na konci

In [43]:
preds["den"] = preds["den"].str.strip()

In [44]:
novy_propojeny_df = pandas.merge(propojeny_df, preds, on=["den"], how="left")
novy_propojeny_df

,cisloStudenta,predmet,znamka,den,mistnost,jmeno_x,datum,jmeno_y
0,2,Dějepis,3.0,pá,u202,Lukáš Jurčík,24.5.2019,Alena Pniáčková
1,2,Společenské vědy,2.0,pá,u202,Lukáš Jurčík,24.5.2019,Alena Pniáčková
2,3,Matematika,2.0,út,u202,Pavel Horák,21.5.2019,Marie Zuzaňáková
3,3,Chemie,5.0,út,u202,Pavel Horák,21.5.2019,Marie Zuzaňáková
4,4,Biologie,1.0,pá,u202,Pavel Kysilka,24.5.2019,Alena Pniáčková
5,5,Dějepis,1.0,po,u202,Kateřina Novotná,20.5.2019,Marie Zuzaňáková
6,6,Fyzika,2.0,čt,u202,Marie Krejcárková,23.5.2019,Petr Ortinský
7,7,Dějepis,4.0,po,u202,Vasil Lácha,20.5.2019,Marie Zuzaňáková
8,8,Matematika,2.0,po,u202,Alexey Opatrný,20.5.2019,Marie Zuzaňáková
9,10,Chemie,2.0,st,u202,Miroslav Bednář,22.5.2019,Petr Ortinský


**Přejmenování sloupců**

In [45]:
novy_propojeny_df = novy_propojeny_df.rename(
    columns={"jmeno_x": "jmeno", "jmeno_y": "predseda"}
)
novy_propojeny_df

,cisloStudenta,predmet,znamka,den,mistnost,jmeno,datum,predseda
0,2,Dějepis,3.0,pá,u202,Lukáš Jurčík,24.5.2019,Alena Pniáčková
1,2,Společenské vědy,2.0,pá,u202,Lukáš Jurčík,24.5.2019,Alena Pniáčková
2,3,Matematika,2.0,út,u202,Pavel Horák,21.5.2019,Marie Zuzaňáková
3,3,Chemie,5.0,út,u202,Pavel Horák,21.5.2019,Marie Zuzaňáková
4,4,Biologie,1.0,pá,u202,Pavel Kysilka,24.5.2019,Alena Pniáčková
5,5,Dějepis,1.0,po,u202,Kateřina Novotná,20.5.2019,Marie Zuzaňáková
6,6,Fyzika,2.0,čt,u202,Marie Krejcárková,23.5.2019,Petr Ortinský
7,7,Dějepis,4.0,po,u202,Vasil Lácha,20.5.2019,Marie Zuzaňáková
8,8,Matematika,2.0,po,u202,Alexey Opatrný,20.5.2019,Marie Zuzaňáková
9,10,Chemie,2.0,st,u202,Miroslav Bednář,22.5.2019,Petr Ortinský
